# Quickstart — Your First Optimization

Submit a DSPy optimization job, monitor it, and test the optimized program.

**Dataset:** GSM8K — grade school math word problems requiring multi-step reasoning.

**What you'll learn:**
1. Connect to the Skynet service
2. Define a signature and metric
3. Submit an optimization job
4. Monitor progress and view results
5. Load and use the optimized program

**Prerequisites:** Skynet service running (`cd backend && python main.py`), `LM_API_KEY` (or `OPENAI_API_KEY`) set.

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

## 1. Configuration

Point the client at your Skynet instance and configure the LLM.

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
LM_BASE_URL = os.getenv("DSPY_LM_BASE_URL", "https://api.openai.com/v1")

# LM_API_KEY is the canonical name; OPENAI_API_KEY is accepted as a
# fallback for backwards-compat and the public OpenAI dev path.
LM_API_KEY = os.getenv("LM_API_KEY") or os.getenv("OPENAI_API_KEY")
if not LM_API_KEY:
    raise ValueError("Set LM_API_KEY (or OPENAI_API_KEY) — any non-empty token works for self-hosted gateways.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": LM_BASE_URL,
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": LM_API_KEY},
}

# Configure dspy for local testing of the optimized program later
dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_base=LM_BASE_URL,
        api_key=LM_API_KEY,
    )
)

## 2. Connect & Health Check

In [ ]:
client = DSPyServiceClient(BASE_URL)
client.health()

In [ ]:
# Check the worker queue before submitting
client.queue()

## 3. Load Dataset

In [ ]:
with open(Path("../../data/gsm8k.json")) as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples")
print(f"Sample: {dataset[0]}")

## 4. Define Signature & Metric

A **signature** declares what the LLM should do (inputs → outputs).  
A **metric** scores each prediction so the optimizer knows what's working.

For GEPA, the metric must return `dspy.Prediction(score=..., feedback=...)` so the optimizer can reflect on failures.

In [ ]:
class MathReasoning(dspy.Signature):
    """Solve grade school math word problems step by step."""
    question: str = dspy.InputField(desc="A math word problem requiring multi-step reasoning")
    answer: str = dspy.OutputField(desc="The final numeric answer")


SIGNATURE_CODE = serialize_source(MathReasoning)
print(SIGNATURE_CODE)

In [ ]:
# The metric body is shipped to the server as a string and exec()'d in
# a fresh namespace; helper imports (re, math, etc.) must therefore live
# inside the function body even though that breaks PEP 8 locally.
# Sending a string also sidesteps inspect.getsource() issues in containers.


METRIC_CODE = '''
def gsm8k_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    import re

    def extract_number(text):
        if not text:
            return ""
        numbers = re.findall(r'-?[\\d,]+\\.?\\d*', text.replace(',', ''))
        return numbers[-1] if numbers else text.strip()

    expected = extract_number(gold.answer or "")
    actual = extract_number(pred.answer or "")

    if expected == actual:
        return dspy.Prediction(score=1.0, feedback="Correct answer.")
    else:
        feedback = f"Incorrect. Expected {gold.answer}, got {pred.answer}. Check each arithmetic step."
        return dspy.Prediction(score=0.0, feedback=feedback)
'''


## 5. Submit Optimization

Build the payload and submit. The service validates everything synchronously, then runs the optimization in the background.

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "light", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"question": "question"},
        "outputs": {"answer": "answer"},
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "model_config": MODEL_CONFIG,
    "reflection_model_config": MODEL_CONFIG,
}

optimization_id = client.submit(payload)
print(f"Submitted: {optimization_id}")

## 6. Monitor Progress

Poll until the job finishes. You'll see status updates, optimizer progress, and log entries.

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=5)

print(f"\nFinal status: {result['status']}")

## 7. View Results

In [ ]:
if result["status"] == "success":
    r = result["result"]
    print(f"Baseline score:  {r.get('baseline_test_metric', 'N/A')}")
    print(f"Optimized score: {r.get('optimized_test_metric', 'N/A')}")
    print(f"Runtime: {r.get('runtime_seconds', 0):.0f}s")
else:
    print(f"Failed: {result.get('message')}")

In [ ]:
# Compact summary card (what the dashboard shows)
client.summary(optimization_id)

In [ ]:
# Recent logs
logs = client.logs(optimization_id, limit=5)
for log in logs:
    print(f"  [{log['level']}] {log['message'][:80]}")

## 8. Load & Test the Optimized Program

Download the artifact and run inference locally.

In [ ]:
program = client.load_program(optimization_id)
print(f"Loaded: {type(program).__name__}")

In [ ]:
test_questions = [
    "A bakery sells 24 cupcakes in the morning and 36 in the afternoon. Each costs $3. How much money did the bakery make?",
    "Lisa has 48 stickers. She gives 1/4 to her friend, then buys 12 more. How many does she have?",
    "A train travels at 60 mph for 2 hours, then 80 mph for 1.5 hours. What is the total distance?",
]

for q in test_questions:
    response = program(question=q)
    print(f"Q: {q}")
    print(f"A: {response.answer}\n")

## 9. Managing Jobs

List, cancel, and delete optimizations.

In [ ]:
# List your recent optimizations
jobs = client.list_optimizations(username="notebook-user", limit=5)
print(f"Total: {jobs['total']}")
for item in jobs["items"]:
    print(f"  {item['optimization_id'][:8]}... | {item['status']} | {item.get('optimizer_name', '')}")

In [ ]:
# Cancel a running job (no-op if already finished)
# client.cancel(optimization_id)

In [ ]:
# Delete a completed job (uncomment to run)
# client.delete(optimization_id)

## What's Next

- **[02_grid_search.ipynb](02_grid_search.ipynb)** — Compare multiple models on the same task using grid search.
- **[03_creative_tasks.ipynb](03_creative_tasks.ipynb)** — Optimize multi-field signatures for creative generation tasks.
- **API Reference** — Visit `http://localhost:8000/reference` for the full interactive API docs.